# Daily Challenge: Fine-Tuning LLMs with LoRA (PEFT)
## Week 7 — Day 4 | DI GenAI & Machine Learning Bootcamp 2026

**Objectives:**
- Apply Low-Rank Adaptation (LoRA) to a pretrained causal LM (`bigscience/bloomz-560m`)
- Fine-tune only the LoRA adapter parameters on the English Quotes dataset
- Save the LoRA adapter weights and reload them with `PeftModel.from_pretrained`
- Generate new text with the fine-tuned model

**Steps:**
1. Install libraries and setup
2. Load model, tokenizer, and dataset
3. Configure and apply LoRA
4. Train with `Trainer`
5. Save the adapter
6. Load for inference and generate text

In [ ]:
%pip install peft==0.4.0 datasets transformers torch --quiet

In [ ]:
import os
import time
import warnings
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import peft
from peft import LoraConfig, get_peft_model, PeftModel
warnings.filterwarnings("ignore")

# Output cache directory (mirrors the notebook hint)
os.makedirs("../cache/working", exist_ok=True)

print(f"PyTorch      {torch.__version__}")
print(f"Transformers {transformers.__version__}")
print(f"PEFT         {peft.__version__}")

## Step 1 — Load the Pretrained Model and Tokenizer

We use **`bigscience/bloomz-560m`** — a 560M-parameter multilingual causal LM from BigScience that supports instruction following. It is small enough to fine-tune on CPU while still being representative of the LoRA workflow used on 7B+ models.

In [ ]:
model_name = "bigscience/bloomz-560m"

print(f"Loading tokenizer from '{model_name}'…")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loading foundation model from '{model_name}'…")
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

n_params = sum(p.numel() for p in foundation_model.parameters())
print(f"\nModel loaded ✓")
print(f"  Total parameters : {n_params:,}")
print(f"  Tokenizer vocab  : {tokenizer.vocab_size:,}")
print(f"  Model type       : {foundation_model.config.model_type}")

## Step 2 — Load and Preprocess the Dataset

The **`Abirate/english_quotes`** dataset contains famous quotes with their authors and tags.  
We take a **10% sample** of the training split and tokenize the `"quote"` column.

In [ ]:
# Load and sample 10% of the training split
full_data = load_dataset("Abirate/english_quotes", split="train")
sample_size = max(1, len(full_data) // 10)
data = full_data.shuffle(seed=42).select(range(sample_size))

print(f"Full dataset  : {len(full_data):,} quotes")
print(f"10% sample    : {len(data):,} quotes")
print(f"\nColumns : {data.column_names}")
print(f"\nSample quotes:")
for i in range(3):
    print(f"  [{i}] {data[i]['quote'][:80]}…")

# Tokenize — map each quote through the tokenizer
data = data.map(
    lambda samples: tokenizer(samples["quote"]),
    batched=True,
)

# Take just 5 examples for the quick training demo (CPU-friendly)
train_sample = data.select(range(5))

print(f"\nTokenized dataset columns : {data.column_names}")
print(f"\nFirst training example:")
print(f"  quote    : {train_sample[0]['quote']}")
print(f"  input_ids: {train_sample[0]['input_ids'][:15]}… (len={len(train_sample[0]['input_ids'])})")

# Display as a table
try:
    from IPython.display import display
    display(train_sample.to_pandas()[["quote", "input_ids"]].head())
except Exception:
    print(train_sample.to_pandas()[["quote"]].head().to_string())

## Step 3 — Configure LoRA and Apply to the Model

### Why `query_key_value` for BLOOM?

BLOOM fuses Q, K, V into a single linear layer called **`query_key_value`**.  
Targeting it with LoRA inserts low-rank adapters into the attention mechanism — the same strategy used in the original LoRA paper for GPT-style models.

### LoRA hyperparameters

| Parameter | Value | Meaning |
|---|---|---|
| `r` | 1 | Rank — number of columns in A / rows in B; keeps adapter tiny |
| `lora_alpha` | 32 | Scaling factor for the LoRA update; effective update = α/r × A·B |
| `lora_dropout` | 0.05 | Dropout applied to the LoRA branch during training |
| `bias` | `"none"` | Bias terms of the original layers remain frozen |
| `task_type` | `CAUSAL_LM` | Tells PEFT this is a decoder-only generation task |

In [ ]:
lora_config = LoraConfig(
    r              = 1,                       # low rank — 1 adapter column per head
    lora_alpha     = 32,                      # scaling: effective update = (alpha/r) * A·B
    target_modules = ["query_key_value"],     # BLOOM fuses Q/K/V into one layer
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = "CAUSAL_LM",
)

# Apply LoRA adapters to the frozen foundation model
peft_model = get_peft_model(foundation_model, lora_config)

print("LoRA applied ✓")
print(f"\nLoraConfig:")
print(f"  r              : {lora_config.r}")
print(f"  lora_alpha     : {lora_config.lora_alpha}")
print(f"  target_modules : {lora_config.target_modules}")
print(f"  lora_dropout   : {lora_config.lora_dropout}")
print(f"  bias           : {lora_config.bias}")
print(f"  task_type      : {lora_config.task_type}")

print()
peft_model.print_trainable_parameters()

## Step 4 — Training with the Hugging Face `Trainer`

Key `TrainingArguments`:
- `auto_find_batch_size=True` — automatically halves the batch size if an OOM error occurs
- `learning_rate=3e-2` — higher than standard fine-tuning; LoRA adapters need larger steps since they start at zero
- `use_cpu=True` — ensures CPU training on machines without a GPU
- `DataCollatorForLanguageModeling(mlm=False)` — left-to-right causal LM collation (no masking)

In [ ]:
output_directory = os.path.join("../cache/working", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to            = "none",           # disable W&B / MLflow logging
    output_dir           = output_directory,
    auto_find_batch_size = True,             # auto-halve batch on OOM
    learning_rate        = 3e-2,             # higher LR suits LoRA adapters
    num_train_epochs     = 1,
    use_cpu              = True,
    logging_steps        = 1,
    save_strategy        = "no",
)

trainer = Trainer(
    model         = peft_model,
    args          = training_args,
    train_dataset = train_sample,
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Training arguments:")
print(f"  output_dir           : {training_args.output_dir}")
print(f"  num_train_epochs     : {training_args.num_train_epochs}")
print(f"  learning_rate        : {training_args.learning_rate}")
print(f"  auto_find_batch_size : {training_args.auto_find_batch_size}")
print(f"  use_cpu              : {training_args.use_cpu}")
print(f"\nTraining on {len(train_sample)} examples…")
trainer.train()
print("\nTraining complete ✓")

## Step 5 — Save the LoRA Adapter

`trainer.model.save_pretrained()` saves **only the LoRA adapter weights** (A and B matrices + config) — **not** the full 560M-parameter model. The saved directory contains:
- `adapter_config.json` — LoRA configuration
- `adapter_model.bin` — the tiny adapter weights (~50KB vs ~1GB for the full model)

In [ ]:
time_now       = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

trainer.model.save_pretrained(peft_model_path)

print(f"LoRA adapter saved → {peft_model_path}")
print("\nSaved files:")
for f in sorted(os.listdir(peft_model_path)):
    size_kb = os.path.getsize(os.path.join(peft_model_path, f)) / 1024
    print(f"  {f:<35}  {size_kb:>8.1f} KB")

## Step 6 — Load the Adapter for Inference

`PeftModel.from_pretrained` reloads the saved LoRA adapter on top of the original frozen `foundation_model`. Setting `is_trainable=False` puts all parameters in eval mode — no further gradient tracking.

In [ ]:
# Reload the frozen base model and attach the saved LoRA adapter
loaded_model = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False,    # inference only — no gradient tracking
)
loaded_model.eval()

print(f"LoRA adapter loaded from : {peft_model_path}")
print(f"is_trainable=False → all parameters frozen for inference")

n_trainable = sum(p.numel() for p in loaded_model.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in loaded_model.parameters())
print(f"\nTrainable params : {n_trainable:,}  (should be 0)")
print(f"Total params     : {n_total:,}")

## Step 7 — Generate Text with the Fine-Tuned Model

In [ ]:
prompts = [
    "Two things are infinite: ",
    "The only way to do great work is to ",
    "In the middle of difficulty lies ",
]

print("=== Text Generation with Fine-Tuned LoRA Model ===\n")

for prompt in prompts:
    inputs  = tokenizer(prompt, return_tensors="pt")
    outputs = peft_model.generate(
        input_ids      = inputs["input_ids"],
        max_new_tokens = 40,
        do_sample      = True,
        temperature    = 0.8,
        top_k          = 50,
        top_p          = 0.95,
        pad_token_id   = tokenizer.eos_token_id,
    )
    generated = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    print(f"Prompt    : \"{prompt}\"")
    print(f"Generated : {generated}")
    print("─" * 65)

In [ ]:
# ── Compare: base model vs LoRA fine-tuned model ─────────────────────────────
prompt = "Two things are infinite: "
inputs = tokenizer(prompt, return_tensors="pt")

print("Comparing base model vs LoRA fine-tuned model\n")
print(f"Prompt : \"{prompt}\"")
print("─" * 65)

# Base foundation model (no LoRA adapter)
with torch.no_grad():
    base_out = foundation_model.generate(
        input_ids      = inputs["input_ids"],
        max_new_tokens = 40,
        do_sample      = True,
        temperature    = 0.8,
        top_k          = 50,
        pad_token_id   = tokenizer.eos_token_id,
    )
print(f"Base model  : {tokenizer.batch_decode(base_out, skip_special_tokens=True)[0]}")

# LoRA fine-tuned model
with torch.no_grad():
    lora_out = peft_model.generate(
        input_ids      = inputs["input_ids"],
        max_new_tokens = 40,
        do_sample      = True,
        temperature    = 0.8,
        top_k          = 50,
        pad_token_id   = tokenizer.eos_token_id,
    )
print(f"LoRA model  : {tokenizer.batch_decode(lora_out, skip_special_tokens=True)[0]}")

## Summary & Key Takeaways

| Step | Action | Key point |
|---|---|---|
| 1 | Load `bigscience/bloomz-560m` | 560M params, causal LM, multilingual |
| 2 | Load `Abirate/english_quotes` (10%) | Tokenize `"quote"` column; 5 examples for CPU training |
| 3 | `LoraConfig(r=1, target_modules=["query_key_value"])` | BLOOM fuses QKV — one LoRA module per attention layer |
| 4 | `get_peft_model` → `Trainer.train()` | Only LoRA A & B updated; 3e-2 LR; `DataCollatorForLanguageModeling(mlm=False)` |
| 5 | `trainer.model.save_pretrained(path)` | Saves **adapter only** (~KB) not the full model (~GB) |
| 6 | `PeftModel.from_pretrained(base, path, is_trainable=False)` | Reloads adapter on frozen base; ready for inference |
| 7 | `model.generate(input_ids, max_new_tokens=40)` | Quote-style completions shaped by the fine-tuned adapter |

### LoRA Parameter Budget (bloomz-560m)

```
Foundation model  : ~559M parameters   (all frozen)
LoRA adapter      :    ~ꟃ parameters   (only A and B in query_key_value layers)
Trained fraction  : < 0.1%  →  extremely cheap fine-tune
Saved adapter     : ~ KB    vs  ~1GB for the full model
```

### Production Workflow

```
Pretrained LLM (frozen)
        +
LoRA adapter (tiny, task-specific)   ← saved per task
        ↓
PeftModel.from_pretrained()          ← merge at serving time
        ↓
inference — zero extra latency if weights are merged via merge_and_unload()
```